# DNN v2 Architecture R&D — Cross-Candle Indicators + Temporal Context

Compare 3 architectures using 33 features (11 raw snapshot + 22 cross-candle indicators):

1. **ContextMLP** — single-snapshot baseline on all 33 features (tests value of cross-candle indicators alone)
2. **ContextConditionedTCN** — FiLM-conditioned temporal conv (cross-candle indicators modulate temporal processing)
3. **ContextAttention** — FiLM-conditioned self-attention variant

All architectures use `elapsed_pct <= 0.50` cutoff to prevent data leakage.

In [1]:
import json
import sys
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, brier_score_loss, f1_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

sys.path.insert(0, "../..")
from polybot.adapters.dnn_models import ContextAttention, ContextConditionedTCN, ContextMLP

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

RAW_COLS = [
    "btc_price",
    "elapsed_pct",
    "market_volume",
    "up_best_bid",
    "up_best_ask",
    "up_bid_depth",
    "up_ask_depth",
    "down_best_bid",
    "down_best_ask",
    "down_bid_depth",
    "down_ask_depth",
]

CROSS_CANDLE_COLS = [
    "prior_return",
    "consecutive_streak",
    "streak_magnitude",
    "rolling_volatility",
    "candle_momentum",
    "ma_crossover",
    "trend_consistency",
    "reversal_regime",
    "rsi",
    "bollinger_pct_b",
    "stochastic_k",
    "adx",
    "return_autocorrelation",
    "multi_candle_return_3",
    "multi_candle_return_6",
    "regime_score",
    "mean_reversion_signal",
    "prior_reversal_rate",
    "volume_momentum",
    "volume_trend",
    "volume_price_correlation",
    "relative_volume",
]

ALL_COLS = RAW_COLS + CROSS_CANDLE_COLS
print(f"Features: {len(RAW_COLS)} raw + {len(CROSS_CANDLE_COLS)} cross-candle = {len(ALL_COLS)} total")

Features: 11 raw + 22 cross-candle = 33 total


## 1. Load Data & Build Datasets

In [2]:
ELAPSED_CUTOFF = 0.50
SEQ_LEN = 50

rows = []
with open("../../data/latest_features.jsonl") as f:
    for line in f:
        rows.append(json.loads(line))
df = pd.DataFrame(rows)
df["target"] = (df["outcome"] == "UP").astype(int)
df[ALL_COLS] = df[ALL_COLS].fillna(0)

# Filter to mid-candle
df_mid = df[df["elapsed_pct"] <= ELAPSED_CUTOFF].copy()
print(f"Snapshots: {len(df_mid):,} / {len(df):,} (cutoff={ELAPSED_CUTOFF})")

# --- Single-snapshot dataset (33 features) ---
candle_df = df_mid.groupby("candle_id").last().reset_index()
X_all_ss = candle_df[ALL_COLS].values.astype(np.float32)
y_all = candle_df["target"].values.astype(np.float32)

split_idx = int(len(X_all_ss) * 0.8)

scaler_ss = StandardScaler()
X_train_ss = scaler_ss.fit_transform(X_all_ss[:split_idx])
X_test_ss = scaler_ss.transform(X_all_ss[split_idx:])
y_train, y_test = y_all[:split_idx], y_all[split_idx:]
print(f"Single-snapshot: train={len(X_train_ss):,}, test={len(X_test_ss):,}, features={X_train_ss.shape[1]}")

# --- Temporal dataset (seq_len x 33 features) ---
candle_ids = df_mid["candle_id"].unique()
sequences, seq_targets = [], []
for cid in candle_ids:
    group = df_mid[df_mid["candle_id"] == cid]
    feats = group[ALL_COLS].values.astype(np.float32)
    target = int(group["target"].iloc[0])
    if len(feats) < SEQ_LEN:
        pad = np.tile(feats[0:1], (SEQ_LEN - len(feats), 1))
        feats = np.vstack([pad, feats])
    else:
        feats = feats[-SEQ_LEN:]
    sequences.append(feats)
    seq_targets.append(target)

X_seq = np.array(sequences, dtype=np.float32)
y_seq = np.array(seq_targets, dtype=np.float32)

scaler_seq = StandardScaler()
X_seq_flat = X_seq.reshape(-1, len(ALL_COLS))
scaler_seq.fit(X_seq_flat[: split_idx * SEQ_LEN])
X_seq_scaled = scaler_seq.transform(X_seq_flat).reshape(X_seq.shape)

X_seq_train, X_seq_test = X_seq_scaled[:split_idx], X_seq_scaled[split_idx:]
y_seq_train, y_seq_test = y_seq[:split_idx], y_seq[split_idx:]
print(f"Temporal: train={len(X_seq_train):,}, test={len(X_seq_test):,}, shape={X_seq_train.shape}")

Snapshots: 152,844 / 306,620 (cutoff=0.5)
Single-snapshot: train=5,208, test=1,302, features=33


Temporal: train=5,208, test=1,302, shape=(5208, 50, 33)


## 2. Training Harness

In [3]:
def train_model(model, X_train, y_train, X_val, y_val, epochs=50, patience=8, lr=0.001, batch_size=256):
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)
    loss_fn = nn.BCEWithLogitsLoss()
    X_t = torch.tensor(X_train, dtype=torch.float32)
    y_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    X_v = torch.tensor(X_val, dtype=torch.float32)
    y_v = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
    loader = DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=True)
    best_vl, best_state, no_improve = float("inf"), None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        for bx, by in loader:
            optimizer.zero_grad()
            loss_fn(model(bx), by).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        model.eval()
        with torch.no_grad():
            vl = loss_fn(model(X_v), y_v).item()
        if vl < best_vl:
            best_vl = vl
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        scheduler.step()
        if no_improve >= patience:
            print(f"  Early stop at epoch {epoch} (best={best_vl:.4f})")
            break
    if best_state:
        model.load_state_dict(best_state)
    return best_vl


def evaluate_model(model, X_test, y_test):
    model.eval()
    X_t = torch.tensor(X_test, dtype=torch.float32)
    for _ in range(10):
        with torch.no_grad():
            model(X_t[:1])
    t0 = time.perf_counter()
    for _ in range(100):
        with torch.no_grad():
            model(X_t[:1])
    latency_ms = (time.perf_counter() - t0) / 100 * 1000
    with torch.no_grad():
        probs = torch.sigmoid(model(X_t)).numpy().flatten()
    preds = (probs >= 0.5).astype(int)
    return {
        "accuracy": accuracy_score(y_test, preds),
        "brier": brier_score_loss(y_test, probs),
        "f1": f1_score(y_test, preds, zero_division=0),
        "latency_ms": latency_ms,
        "probs": probs,
    }


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

## 3. Architecture A: ContextMLP (single-snapshot, 33 features)

Baseline: does adding cross-candle indicators improve over v1's 11-feature ResidualMLP?

In [4]:
m1 = ContextMLP(input_size=33)
print(f"ContextMLP: {count_params(m1):,} params")
t0 = time.perf_counter()
train_model(m1, X_train_ss, y_train, X_test_ss, y_test)
t1 = time.perf_counter() - t0
r1 = evaluate_model(m1, X_test_ss, y_test)
print(
    f"  Time={t1:.1f}s  Acc={r1['accuracy'] * 100:.1f}%  Brier={r1['brier']:.4f}  F1={r1['f1'] * 100:.1f}%  Lat={r1['latency_ms']:.2f}ms"
)

ContextMLP: 138,881 params


  Early stop at epoch 15 (best=0.5541)
  Time=1.9s  Acc=72.3%  Brier=0.1864  F1=71.8%  Lat=0.09ms


## 4. Architecture B: ContextConditionedTCN (FiLM + temporal conv)

Primary candidate. Cross-candle indicators modulate temporal processing of raw snapshot sequences via FiLM conditioning.

In [5]:
m2 = ContextConditionedTCN(raw_size=11, context_size=22)
print(f"ContextConditionedTCN: {count_params(m2):,} params")
t0 = time.perf_counter()
train_model(m2, X_seq_train, y_seq_train, X_seq_test, y_seq_test)
t2 = time.perf_counter() - t0
r2 = evaluate_model(m2, X_seq_test, y_seq_test)
print(
    f"  Time={t2:.1f}s  Acc={r2['accuracy'] * 100:.1f}%  Brier={r2['brier']:.4f}  F1={r2['f1'] * 100:.1f}%  Lat={r2['latency_ms']:.2f}ms"
)

ContextConditionedTCN: 94,081 params


  Early stop at epoch 13 (best=0.5667)


  Time=41.7s  Acc=69.9%  Brier=0.1925  F1=67.1%  Lat=0.41ms


## 5. Architecture C: ContextAttention (FiLM + self-attention)

In [6]:
m3 = ContextAttention(raw_size=11, context_size=22)
print(f"ContextAttention: {count_params(m3):,} params")
t0 = time.perf_counter()
train_model(m3, X_seq_train, y_seq_train, X_seq_test, y_seq_test)
t3 = time.perf_counter() - t0
r3 = evaluate_model(m3, X_seq_test, y_seq_test)
print(
    f"  Time={t3:.1f}s  Acc={r3['accuracy'] * 100:.1f}%  Brier={r3['brier']:.4f}  F1={r3['f1'] * 100:.1f}%  Lat={r3['latency_ms']:.2f}ms"
)

ContextAttention: 89,345 params


  Early stop at epoch 12 (best=0.5522)
  Time=16.3s  Acc=71.1%  Brier=0.1861  F1=71.9%  Lat=0.27ms


## 6. Comparison & Winner Selection

In [7]:
results = pd.DataFrame(
    [
        {
            "Arch": "ContextMLP",
            "Type": "single",
            "Params": count_params(m1),
            "Time(s)": round(t1, 1),
            "Lat(ms)": round(r1["latency_ms"], 2),
            "Acc%": round(r1["accuracy"] * 100, 1),
            "Brier": round(r1["brier"], 4),
            "F1%": round(r1["f1"] * 100, 1),
        },
        {
            "Arch": "CtxCondTCN",
            "Type": "temporal",
            "Params": count_params(m2),
            "Time(s)": round(t2, 1),
            "Lat(ms)": round(r2["latency_ms"], 2),
            "Acc%": round(r2["accuracy"] * 100, 1),
            "Brier": round(r2["brier"], 4),
            "F1%": round(r2["f1"] * 100, 1),
        },
        {
            "Arch": "CtxAttention",
            "Type": "temporal",
            "Params": count_params(m3),
            "Time(s)": round(t3, 1),
            "Lat(ms)": round(r3["latency_ms"], 2),
            "Acc%": round(r3["accuracy"] * 100, 1),
            "Brier": round(r3["brier"], 4),
            "F1%": round(r3["f1"] * 100, 1),
        },
    ]
)
print(results.to_string(index=False))

# Select winner by Brier (lower = better), tiebreak by latency
candidates = [
    ("ContextMLP", r1, "single", count_params(m1), t1),
    ("ContextConditionedTCN", r2, "temporal", count_params(m2), t2),
    ("ContextAttention", r3, "temporal", count_params(m3), t3),
]
candidates.sort(key=lambda x: x[1]["brier"])
best = candidates[0]
second = candidates[1]

if abs(best[1]["brier"] - second[1]["brier"]) < 0.005 and second[1]["latency_ms"] < best[1]["latency_ms"]:
    print(f"\nBrier within 0.005, switching to {second[0]} (faster)")
    best = second

print(f"\n{'=' * 50}")
print(f"  WINNER: {best[0]}")
print(f"  Type: {best[2]}, Params: {best[3]:,}")
print(f"  Brier={best[1]['brier']:.4f}, Acc={best[1]['accuracy'] * 100:.1f}%, Lat={best[1]['latency_ms']:.2f}ms")
print(f"  temporal={best[2] == 'temporal'}")
print(f"{'=' * 50}")

        Arch     Type  Params  Time(s)  Lat(ms)  Acc%  Brier  F1%
  ContextMLP   single  138881      1.9     0.09  72.3 0.1864 71.8
  CtxCondTCN temporal   94081     41.7     0.41  69.9 0.1925 67.1
CtxAttention temporal   89345     16.3     0.27  71.1 0.1861 71.9

Brier within 0.005, switching to ContextMLP (faster)

  WINNER: ContextMLP
  Type: single, Params: 138,881
  Brier=0.1864, Acc=72.3%, Lat=0.09ms
  temporal=False
